# Week 7 — Policy Gradients and REINFORCE · Workbook

This workbook has two explicit exercise modes, controlled by `REFERENCE_MODE`
in the setup cell.

- **Reference (`True`, default):** supplied implementations demonstrate the
  calculations. Passing these checks verifies the reference, not your code.
- **Learner (`False`):** complete each scaffold's function, then restart and
  run all cells. The corresponding checks call your function. An unfinished
  scaffold raises `NotImplementedError`; there is no reference fallback.
  In a reference-wrapper scaffold, replace its guarded placeholder body with
  your implementation before choosing learner mode.

Change the setup flag before restarting. Record a boundary-case prediction
before each check and change an input after it passes. Later experiment cells
without a scaffold are reference demonstrations, not automated learner grades.

**Teaching companion:** [Worked examples, assumptions and self-checks](teaching_guide.md).
For additional independently graded tasks, see the
[progressive coding projects](../docs/INTERACTIVE_LEARNING.md).


**A bounded first investigation:** Plot the gradient estimator, not just its loss. Start with **Exercises 1, 2, 4 and 5** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-plot-the-gradient-estimator-not-just-its-loss).
Save your own prediction before running the reference cells. For your own implementation, choose learner mode in setup, restart, and verify
that the corresponding check calls your completed function.


In [ ]:
import sys, os
# Make the repo root and this week's project package importable.
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, torch, matplotlib.pyplot as plt
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
from project import solutions as sol
from project.agent import REINFORCEAgent
from project.train import train

# True runs documented references. False checks YOUR completed functions.
# Restart and run all after changing this flag; an unfinished learner stops.
REFERENCE_MODE = True
def select_exercise(reference, learner):
    return reference if REFERENCE_MODE else learner
print("Exercise mode:", "reference demonstration" if REFERENCE_MODE else "learner checks (no fallback)")


## Exercise 1 (★) — Verify the log-derivative identity

Show empirically that $\nabla_\theta p_\theta(x) = p_\theta(x) \nabla_\theta \log p_\theta(x)$
for $p_\theta(x) = \mathcal{N}(x;\theta,1)$ by finite differences at $x=0.3$, $\theta=-0.2$.

> **Hint**: $\log p_\theta(x) = -\tfrac{1}{2}(x-\theta)^2 + \text{const}$, so $\nabla_\theta \log p_\theta(x) = (x-\theta)$.


In [ ]:
def my_log_prob_normal(x, mu, sigma=1.0):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement my_log_prob_normal before running learner checks")
    try:
        raise NotImplementedError
    except NotImplementedError:
        return sol.log_prob_normal(x, mu, sigma)


In [ ]:
# Checks the selected implementation against independently calculated expected values.
ref = select_exercise(sol.log_prob_normal, my_log_prob_normal)(0.3, -0.2)
tref = torch.distributions.Normal(-0.2, 1.0).log_prob(torch.tensor(0.3)).item()
assert abs(ref - tref) < 1e-6, (ref, tref)
print('E1 ok:', ref)


## Exercise 2 (★) — Score function has zero mean

Sample $a \sim \mathrm{Categorical}(\mathrm{softmax}(\ell))$ with $\ell \in \mathbb{R}^4$ and
show by Monte Carlo that $\mathbb{E}_a[\nabla_\ell \log \pi(a)] \to 0$ as $N \to \infty$.

> **Hint**: use `torch.distributions.Categorical(logits=...)` and `dist.log_prob(a).backward()`
> for a per-sample logits gradient, then average.


In [ ]:
torch.manual_seed(0)
logits = torch.randn(4, requires_grad=True)
dist = torch.distributions.Categorical(logits=logits)
N = 20000
grads = torch.zeros(4)
for _ in range(N):
    a = dist.sample()
    g, = torch.autograd.grad(dist.log_prob(a), logits, retain_graph=True)
    grads += g
avg = grads / N
print('mean score:', avg.detach().numpy())
assert avg.abs().max().item() < 0.05


In [ ]:
# Graded: nothing to do; the cell above is the assertion.
print('E2 ok')


## Exercise 3 (★★) — `discount_cumsum`

Implement `discount_cumsum(rews, gamma)` in $O(T)$ and verify $G_t = r_t + \gamma G_{t+1}$.

> **Hint**: iterate from the end.


In [ ]:
def my_discount_cumsum(rews, gamma):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement my_discount_cumsum before running learner checks")
    try:
        raise NotImplementedError
    except NotImplementedError:
        return sol.discount_cumsum(rews, gamma)


In [ ]:
G = select_exercise(sol.discount_cumsum, my_discount_cumsum)(np.array([1.0, 1.0, 1.0]), 0.5)
assert np.allclose(G, [1.75, 1.5, 1.0])
print('E3 ok:', G)


## Exercise 4 (★★) — REINFORCE on a two-armed bandit

Build a single-state policy $\pi_\theta(a=1) = \sigma(\theta)$ and train it to prefer arm 1
(reward 1 vs.\ 0). Reach $\sigma(\theta) > 0.95$ within 200 updates.

> **Hint**: `reinforce_loss(logp, returns)` from `project.solutions` is exactly the loss.


In [ ]:
theta = torch.tensor(0.0, requires_grad=True)
opt = torch.optim.Adam([theta], lr=0.05)
for step in range(200):
    p = torch.sigmoid(theta)
    dist = torch.distributions.Bernoulli(p)
    a = dist.sample()
    r = a  # reward = 1 if arm 1, else 0
    loss = sol.reinforce_loss(dist.log_prob(a).unsqueeze(0), r.unsqueeze(0))
    opt.zero_grad(); loss.backward(); opt.step()
p_final = torch.sigmoid(theta).item()
print('final p(arm=1) =', p_final)
assert p_final > 0.9, p_final


## Exercise 5 (★★) — Variance reduction with $V_\phi$

Train two CartPole agents (with and without baseline) for 80 episodes and plot the loss-contribution variance returned by the project. It is a proxy, not the variance of per-sample parameter gradients.

> **Hint**: use `train(seed=0, n_episodes=80, use_baseline=...)` and read `out.estimator_variances`.


In [ ]:
out_b = train(seed=0, n_episodes=80, use_baseline=True)
out_n = train(seed=0, n_episodes=80, use_baseline=False)
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(out_n.estimator_variances, color=RL_PALETTE['blue'], label='no baseline')
ax.plot(out_b.estimator_variances, color=RL_PALETTE['teal'], label='+ V_phi')
ax.set_yscale('log'); ax.set_xlabel('episode'); ax.set_ylabel('Var[g]'); ax.legend()
fig.tight_layout(); plt.show()
# The baseline variant should have a lower median variance.
import numpy as np
assert np.isfinite(out_b.estimator_variances).all() and np.isfinite(out_n.estimator_variances).all()
print('Observed baseline/no-baseline medians:', np.median(out_b.estimator_variances), np.median(out_n.estimator_variances))


## Exercise 6 (★★) — CartPole convergence

Train REINFORCE+baseline to a 20-episode-window mean of $\ge 100$ within 250 episodes.
We use 100 (rather than the 195 of the lecture) as a *workbook-friendly* threshold
so the cell runs quickly.


In [ ]:
out = train(seed=0, n_episodes=250, use_baseline=True)
import numpy as np
last = np.mean(out.returns[-20:])
print('last-20 mean:', last, 'solved_at:', out.solved_at)
assert np.isfinite(last)
print('This is one training seed; independent evaluation is needed for a performance comparison.')


## Exercise 7 (★★★) — Top-$K$ off-policy correction

Implement the multiplier $\lambda_K(s,a) = K(1 - \pi_\theta(a|s))^{K-1}$ and derive it as the derivative of inclusion probability $1-(1-p)^K$ under $K$ independent draws with replacement. Explain why this is not a general correction for arbitrary top-K ranking or logging-policy mismatch.

> **Hint**: compare gradient direction with and without $\lambda_K$ on a 5-arm synthetic env.


In [ ]:
p = np.array([0.5, 0.2, 0.15, 0.1, 0.05])
K = 3
lam = sol.topk_offpolicy_multiplier(p, K)
print('lambda_K =', np.round(lam, 3))
# Sanity: large p -> small multiplier (we already explore it via top-K); small p -> large.
assert lam[0] < lam[-1]


## Exercise 8 (★★★, *meaningful objective*) — SNIPS for a synthetic insurance-style router

Logged $(\pi_b, a, r)$ data from an $\varepsilon$-greedy lead-broker router. Estimate the
value of a deterministic argmax target policy using **Self-Normalised IPS**, and contrast
with the direct method.

> **Hint**: Self-normalisation rescales weights; it does not clip them. The reference helper subtracts the maximum log-ratio before exponentiating; this common scaling cancels exactly and avoids changing the estimator.


In [ ]:
rng = np.random.default_rng(2)
n = 5000
# behaviour: epsilon-greedy, eps=0.3, 4 brokers; broker 0 is truly best
eps = 0.3; n_brokers = 4
probs_b = np.full(n_brokers, eps / n_brokers); probs_b[0] += 1 - eps
log_probs_b = np.log(np.clip(probs_b, 1e-9, 1.0))
acts = rng.choice(n_brokers, size=n, p=probs_b)
true_q = np.array([0.6, 0.4, 0.35, 0.25])
rews = rng.binomial(1, true_q[acts]).astype(float)
# target: greedy on broker 0 (deterministic; eps_target=1e-3 for numerical safety)
et = 1e-3; probs_t = np.full(n_brokers, et / n_brokers); probs_t[0] += 1 - et
log_probs_t = np.log(probs_t[acts])
log_probs_b_sel = log_probs_b[acts]
v_snips = sol.snips_estimator(log_probs_b_sel, log_probs_t, rews)
print('SNIPS value of greedy target:', round(v_snips, 3), 'true:', true_q[0])
# Direct method: just average reward of broker-0 actions in the logs.
mask = acts == 0
v_dm = rews[mask].mean() if mask.any() else float('nan')
print('Direct method:', round(float(v_dm), 3))
assert abs(v_snips - true_q[0]) < 0.05


### Recap

You derived REINFORCE, demonstrated the baseline lemma empirically, trained on CartPole,
and applied the YouTube top-$K$ correction + SNIPS to a matching-flavoured logged dataset.
Next week: A2C, GAE, and TRPO replace the Monte-Carlo $G_t$ with bootstrapped multi-step
estimates and impose a trust region on the policy update.


<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 (★) — Verify the log-derivative identity</summary>

> **Hint**: $\log p_\theta(x) = -\tfrac{1}{2}(x-\theta)^2 + \text{const}$, so $\nabla_\theta \log p_\theta(x) = (x-\theta)$.

</details>

<details>
<summary>Exercise 2 (★) — Score function has zero mean</summary>

> **Hint**: use `torch.distributions.Categorical(logits=...)` and `dist.log_prob(a).backward()`
> for a per-sample logits gradient, then average.

</details>

<details>
<summary>Exercise 3 (★★) — `discount_cumsum`</summary>

> **Hint**: iterate from the end.

</details>

<details>
<summary>Exercise 4 (★★) — REINFORCE on a two-armed bandit</summary>

> **Hint**: `reinforce_loss(logp, returns)` from `project.solutions` is exactly the loss.

</details>

<details>
<summary>Exercise 5 (★★) — Variance reduction with $V_\phi$</summary>

> **Hint**: use `train(seed=0, n_episodes=80, use_baseline=...)` and read `out.estimator_variances`.

</details>

<details>
<summary>Exercise 6 (★★) — CartPole convergence</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 7 (★★★) — Top-$K$ off-policy correction</summary>

> **Hint**: compare gradient direction with and without $\lambda_K$ on a 5-arm synthetic env.

</details>

<details>
<summary>Exercise 8 (★★★, *meaningful objective*) — SNIPS for a synthetic insurance-style router</summary>

> **Hint**: Self-normalisation rescales weights; it does not clip them. The reference helper subtracts the maximum log-ratio before exponentiating; this common scaling cancels exactly and avoids changing the estimator.

</details>

After solving, change one input or assumption and predict the result before
running it. In reference mode, passing checks verifies the demonstration. In learner mode,
the selected checks must pass against your completed functions without fallback.


## Baseline calculation: connect the guide to the helper
Read the [three-depth explanation](teaching_guide.md#choose-your-explanation-depth-why-the-best-baseline-is-a-weighted-mean), predict 4.4, then check the score-weighted baseline. This deterministic example complements Exercise 5; it does not claim a universal variance reduction in training.

In [ ]:
weights = np.array([1.0, 4.0])
returns = np.array([2.0, 5.0])
baseline = sol.optimal_baseline(weights, returns)
assert np.isclose(baseline, 4.4)
print("weighted baseline:", baseline)
print("weighted residual sum:", np.sum(weights * (returns - baseline) ** 2))
print("ordinary-mean residual sum:", np.sum(weights * (returns - returns.mean()) ** 2))

## Statistical investigation — A baseline changes gradient variance, not the objective

**Goal.** Derive how an action-independent baseline changes gradient variance. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** For a Bernoulli policy with p=0.25 and reward A, the logit score is A-p.
Compute E[(A-p)(A-b)] before running for b=0, b=p, and b=1-p. An action-independent
constant baseline has zero expected score contribution. Which baseline
minimizes variance in this particular one-step problem?

**Work a small case by hand.** Here p=0.25 and the logit score is A-p. With baseline b, the two possible
gradient samples are 0.25b when A=0 and 0.75(1-b) when A=1. Their mean is
0.75(0.25b)+0.25(0.75(1-b))=0.1875=p(1-p), independent of b.
The variance vanishes when the two samples coincide:
0.25b=0.75(1-b), hence b=0.75=1-p. The ordinary reward mean p=0.25 is
not this score-weighted variance-minimizing baseline.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/02-expectation-variance.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
p=.25;actions=np.array([0.,1.]);prob=np.array([1-p,p]);score=actions-p
baselines=np.linspace(-.1,1.1,100)
means=[];variances=[]
for b in baselines:
    g=score*(actions-b);m=prob@g
    means.append(m);variances.append(prob@((g-m)**2))
assert np.allclose(means,p*(1-p))
rng_stats=np.random.default_rng(107);a=rng_stats.binomial(1,p,3000)
for b in [0,p,1-p]:
    g=(a-p)*(a-b)
    print('baseline',b,'sample gradient mean',g.mean(),'sample variance',g.var())
fig,axes=plt.subplots(1,2,figsize=(9,3.3))
axes[0].plot(baselines,means);axes[0].axhline(p*(1-p),color=RL_PALETTE['red'],ls='--')
axes[0].set(xlabel='Action-independent baseline b',ylabel='Exact expected gradient')
axes[1].plot(baselines,variances);axes[1].axvline(1-p,color=RL_PALETTE['teal'],ls='--',label='Variance minimum: 1-p')
axes[1].set(xlabel='Baseline b',ylabel='Exact gradient variance');axes[1].legend()
fig.tight_layout();plt.show()


**Interpret and transfer.** The variance minimum is score-weighted; it need not equal the ordinary
expected return p. A learned state baseline needs to remain independent of
the sampled action conditional on state for the usual identity.

**Transfer.** Make the baseline b(A)=A. The estimator becomes zero and is
biased unless the true gradient is zero. Explain exactly which conditional
independence assumption you broke.


**Read your result.** Read both panels together: expected gradient stays constant while its
variance changes. At b=0.75 the printed samples are all 0.1875 in this
special one-step problem; a learned general-purpose baseline need not achieve that.

<details><summary>Worked feedback for the changed assumption</summary>

Choosing b(A)=A makes every gradient sample (A-p)(A-A)=0. The true gradient
is still 0.1875, so this estimator is biased. The baseline now depends on the
sampled action; E[score×baseline] is no longer zero. Its zero variance is
therefore not evidence of a correct estimator.

</details>
